# 23 — Create Monthly Satellite Data

Creates monthly Sentinel-1 and Sentinel-2 composites directly from the acquisition-level TIFFs produced by Notebook 10.

**Source:** `finals/daily_datasets/`  
**Output:** `finals/monthly_datasets/`

Important: September 2024 is split at **September 27** so a composite never mixes pre- and post-Helene imagery. Thus there are 11 analysis periods.

In [1]:
# 1. Packages
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import rasterio

warnings.filterwarnings("ignore", category=RuntimeWarning)
print("Packages loaded.")

Packages loaded.


In [2]:
# 2. Paths
BASE_DIR = Path("/Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets")
FINALS_DIR = BASE_DIR / "finals"
DAILY_DIR = FINALS_DIR / "daily_datasets"
DAILY_INVENTORY_FILE = DAILY_DIR / "daily_satellite_inventory.csv"
SELECTED_SAMPLE_FILE = DAILY_DIR / "selected_site_sample.csv"

MONTHLY_DIR = FINALS_DIR / "monthly_datasets"
S1_MONTHLY_DIR = MONTHLY_DIR / "sentinel1"
S2_MONTHLY_DIR = MONTHLY_DIR / "sentinel2"

QUALITY_CSV_FILE = MONTHLY_DIR / "monthly_image_quality.csv"
QUALITY_EXCEL_FILE = MONTHLY_DIR / "monthly_image_quality.xlsx"
SUMMARY_CSV_FILE = MONTHLY_DIR / "monthly_quality_summary.csv"
PERIOD_DEFINITION_FILE = MONTHLY_DIR / "monthly_period_definitions.csv"
PERIOD_DIMENSION_FILE = MONTHLY_DIR / "monthly_period_dimension.csv"
SITE_DIMENSION_FILE = MONTHLY_DIR / "monthly_site_dimension.csv"
BUILD_ACTION_FILE = MONTHLY_DIR / "monthly_build_action_summary.csv"
FOLDER_SUMMARY_FILE = MONTHLY_DIR / "monthly_folder_summary.csv"
SAMPLE_VALIDATION_FILE = MONTHLY_DIR / "monthly_sample_validation.csv"

MONTHLY_DIR.mkdir(parents=True, exist_ok=True)
print(MONTHLY_DIR)

/Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/monthly_datasets


In [3]:
# 3. Check Notebook 10 outputs
required = [DAILY_INVENTORY_FILE, SELECTED_SAMPLE_FILE]
missing = [x for x in required if not x.exists()]
if missing:
    raise FileNotFoundError("Missing Notebook 10 outputs:\n" + "\n".join(map(str, missing)))
print("Required source files found.")

Required source files found.


In [4]:
# 4. Study calendar and settings
HELENE_REFERENCE_DATE = pd.Timestamp("2024-09-27")
STUDY_START = pd.Timestamp("2024-05-10")
STUDY_END = pd.Timestamp("2025-02-13")

SKIP_EXISTING_MONTHLY = True
RUN_TEST_ONLY = False

EXPECTED_TREATMENT_SITES = 10
EXPECTED_CONTROLS_PER_TREATMENT = 5

SENSOR_BANDS = {
    "sentinel1": ["VV", "VH", "VV_minus_VH"],
    "sentinel2": ["B2", "B3", "B4", "B8", "B11", "B12", "NDVI", "NDWI"],
}

for root in [S1_MONTHLY_DIR, S2_MONTHLY_DIR]:
    for group in ["treatment", "counterfactual"]:
        for phase in ["before", "after"]:
            (root / group / phase).mkdir(parents=True, exist_ok=True)

print("Settings ready.")

Settings ready.


In [5]:
# 5. Load fixed selected sample
selected_sample = pd.read_csv(SELECTED_SAMPLE_FILE)

required = ["site_id", "group", "matched_treatment_site_id", "control_rank"]
missing = [x for x in required if x not in selected_sample.columns]
if missing:
    raise ValueError(f"Selected sample missing columns: {missing}")

selected_sample["site_id"] = selected_sample["site_id"].astype(str)
selected_sample["group"] = selected_sample["group"].astype(str).str.strip().str.lower()
selected_sample["matched_treatment_site_id"] = selected_sample["matched_treatment_site_id"].astype("string")
selected_sample["control_rank"] = pd.to_numeric(selected_sample["control_rank"], errors="coerce")

m = selected_sample["group"].eq("treatment")
selected_sample.loc[m, "matched_treatment_site_id"] = selected_sample.loc[m, "site_id"].astype(str)
selected_sample = selected_sample.drop_duplicates(["site_id", "group"]).reset_index(drop=True)

treat = selected_sample[selected_sample["group"].eq("treatment")].copy()
controls = selected_sample[selected_sample["group"].eq("counterfactual")].copy()

controls_per_treatment = (
    controls.groupby("matched_treatment_site_id")["site_id"]
    .nunique().rename("counterfactual_count").reset_index()
)

sample_validation = pd.DataFrame({
    "matched_treatment_site_id": treat["site_id"].drop_duplicates().astype(str)
}).merge(controls_per_treatment, on="matched_treatment_site_id", how="left")

sample_validation["counterfactual_count"] = sample_validation["counterfactual_count"].fillna(0).astype(int)
sample_validation["expected_counterfactual_count"] = EXPECTED_CONTROLS_PER_TREATMENT
sample_validation["sample_complete"] = sample_validation["counterfactual_count"].eq(EXPECTED_CONTROLS_PER_TREATMENT)
sample_validation.to_csv(SAMPLE_VALIDATION_FILE, index=False)

print("Treatment:", treat["site_id"].nunique())
print("Controls:", controls["site_id"].nunique())
print("Total:", selected_sample["site_id"].nunique())
print(sample_validation.to_string(index=False))

Treatment: 10
Controls: 50
Total: 60
matched_treatment_site_id  counterfactual_count  expected_counterfactual_count  sample_complete
           treatment_0001                     5                              5             True
           treatment_0002                     5                              5             True
           treatment_0003                     5                              5             True
           treatment_0004                     5                              5             True
           treatment_0005                     5                              5             True
           treatment_0006                     5                              5             True
           treatment_0007                     5                              5             True
           treatment_0008                     5                              5             True
           treatment_0009                     5                              5             True
   

## 6. Monthly period definitions

May and February are partial because the downloaded study window begins May 10 and ends February 13.

September is split into `2024-09_pre` and `2024-09_post`. This preserves the causal cutoff rather than combining observations from both sides of Helene.

In [6]:
# 6. Monthly analysis periods
records = [
    ("before","2024-05","2024-05-10","2024-05-31"),
    ("before","2024-06","2024-06-01","2024-06-30"),
    ("before","2024-07","2024-07-01","2024-07-31"),
    ("before","2024-08","2024-08-01","2024-08-31"),
    ("before","2024-09_pre","2024-09-01","2024-09-26"),
    ("after","2024-09_post","2024-09-27","2024-09-30"),
    ("after","2024-10","2024-10-01","2024-10-31"),
    ("after","2024-11","2024-11-01","2024-11-30"),
    ("after","2024-12","2024-12-01","2024-12-31"),
    ("after","2025-01","2025-01-01","2025-01-31"),
    ("after","2025-02","2025-02-01","2025-02-13"),
]

period_definitions = pd.DataFrame(records, columns=["period","month_id","period_start","period_end"])
period_definitions["period_start"] = pd.to_datetime(period_definitions["period_start"])
period_definitions["period_end"] = pd.to_datetime(period_definitions["period_end"])
period_definitions["period_number"] = np.arange(1, len(period_definitions)+1)
period_definitions["period_id"] = [f"M{i:02d}" for i in period_definitions["period_number"]]
period_definitions["calendar_days"] = (period_definitions["period_end"] - period_definitions["period_start"]).dt.days + 1

cross = (
    (period_definitions["period_start"] < HELENE_REFERENCE_DATE)
    & (period_definitions["period_end"] >= HELENE_REFERENCE_DATE)
)
assert not cross.any()
assert period_definitions.iloc[0]["period_start"] == STUDY_START
assert period_definitions.iloc[-1]["period_end"] == STUDY_END

period_definitions.to_csv(PERIOD_DEFINITION_FILE, index=False)
print(period_definitions.to_string(index=False))

period     month_id period_start period_end  period_number period_id  calendar_days
before      2024-05   2024-05-10 2024-05-31              1       M01             22
before      2024-06   2024-06-01 2024-06-30              2       M02             30
before      2024-07   2024-07-01 2024-07-31              3       M03             31
before      2024-08   2024-08-01 2024-08-31              4       M04             31
before  2024-09_pre   2024-09-01 2024-09-26              5       M05             26
 after 2024-09_post   2024-09-27 2024-09-30              6       M06              4
 after      2024-10   2024-10-01 2024-10-31              7       M07             31
 after      2024-11   2024-11-01 2024-11-30              8       M08             30
 after      2024-12   2024-12-01 2024-12-31              9       M09             31
 after      2025-01   2025-01-01 2025-01-31             10       M10             31
 after      2025-02   2025-02-01 2025-02-13             11       M11        

In [7]:
# 7. Load and clean daily inventory
inventory = pd.read_csv(DAILY_INVENTORY_FILE)
required = ["site_id","group","sensor","acquisition_date","file_path"]
missing = [x for x in required if x not in inventory.columns]
if missing:
    raise ValueError(f"Daily inventory missing columns: {missing}")

inventory["site_id"] = inventory["site_id"].astype(str)
inventory["group"] = inventory["group"].astype(str).str.strip().str.lower()
inventory["sensor"] = inventory["sensor"].astype(str).str.strip().str.lower()
inventory["acquisition_date"] = pd.to_datetime(inventory["acquisition_date"], errors="coerce")
inventory = inventory[inventory["acquisition_date"].notna()].copy()

selected_ids = set(selected_sample["site_id"])
inventory = inventory[inventory["site_id"].isin(selected_ids)].copy()

if "status" in inventory.columns:
    inventory = inventory[inventory["status"].isin(["success","existing"])].copy()

inventory = inventory[inventory["acquisition_date"].between(STUDY_START, STUDY_END)].copy()
inventory = inventory.drop_duplicates(
    ["site_id","group","sensor","acquisition_date","file_path"]
).reset_index(drop=True)

inventory["file_exists"] = inventory["file_path"].astype(str).map(lambda x: Path(x).exists())
print("Missing local source TIFF rows:", (~inventory["file_exists"]).sum())
inventory = inventory[inventory["file_exists"]].copy()

print("Usable acquisitions:", len(inventory))
print(inventory["sensor"].value_counts())

Missing local source TIFF rows: 0
Usable acquisitions: 4081
sensor
sentinel2    2146
sentinel1    1935
Name: count, dtype: int64


In [8]:
# 8. Raster functions
def read_raster(file_path):
    with rasterio.open(file_path) as src:
        return {
            "data": src.read(masked=True).astype("float32").filled(np.nan),
            "profile": src.profile.copy(),
            "transform": src.transform,
            "crs": src.crs,
            "width": src.width,
            "height": src.height,
            "band_count": src.count,
        }

def inspect_tiff(file_path, band_names):
    p = Path(file_path)
    if not p.exists():
        return {"reusable":False,"validation_status":"missing","error":"File missing."}
    try:
        r = read_raster(p)
        if r["band_count"] != len(band_names):
            return {"reusable":False,"validation_status":"wrong_band_count",
                    "error":f"Expected {len(band_names)} bands; found {r['band_count']}."}
        if r["width"] <= 0 or r["height"] <= 0 or r["crs"] is None:
            return {"reusable":False,"validation_status":"invalid_raster","error":"Invalid dimensions or CRS."}
        finite = np.isfinite(r["data"])
        any_valid = finite.any(axis=0)
        all_valid = finite.all(axis=0)
        result = {
            "reusable":True, "validation_status":"valid", "error":None,
            "valid_pixel_fraction":float(any_valid.mean()),
            "valid_pixel_percentage":float(any_valid.mean()*100),
            "valid_pixel_fraction_any_band":float(any_valid.mean()),
            "valid_pixel_percentage_any_band":float(any_valid.mean()*100),
            "valid_pixel_fraction_all_bands":float(all_valid.mean()),
            "valid_pixel_percentage_all_bands":float(all_valid.mean()*100),
        }
        for i, name in enumerate(band_names):
            f = float(finite[i].mean())
            result[f"valid_fraction_{name}"] = f
            result[f"valid_percentage_{name}"] = f*100
        return result
    except Exception as e:
        return {"reusable":False,"validation_status":"corrupt_or_unreadable","error":str(e)}

def check_compatibility(infos):
    ref = infos[0]
    for i, cur in enumerate(infos[1:], start=2):
        if cur["data"].shape != ref["data"].shape:
            return False, f"Image {i} shape mismatch."
        if str(cur["crs"]) != str(ref["crs"]):
            return False, f"Image {i} CRS mismatch."
        if cur["transform"] != ref["transform"]:
            return False, f"Image {i} pixel-grid mismatch."
    return True, None

def create_monthly_composite(source_files):
    infos = [read_raster(x) for x in source_files]
    ok, error = check_compatibility(infos)
    if not ok:
        raise ValueError(error)
    if len(infos) == 1:
        return infos[0]["data"].copy(), infos[0]
    stack = np.stack([x["data"] for x in infos], axis=0)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        composite = np.nanmedian(stack, axis=0)
    return composite, infos[0]

def save_composite(composite, reference, output_file):
    profile = reference["profile"].copy()
    profile.update(driver="GTiff", dtype="float32",
                   count=composite.shape[0], compress="deflate", nodata=np.nan)
    Path(output_file).parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(output_file, "w", **profile) as dst:
        dst.write(composite.astype("float32"))

def invalid_backup_path(p):
    p = Path(p)
    candidate = p.with_name(p.name + ".invalid")
    n = 1
    while candidate.exists():
        candidate = p.with_name(p.name + f".invalid_{n}")
        n += 1
    return candidate

def quality_label(f):
    if pd.isna(f): return "missing"
    if f >= .80: return "excellent"
    if f >= .50: return "usable"
    if f >= .20: return "limited"
    if f > 0: return "poor"
    return "unusable"

In [9]:
# 9. Resume-safe monthly builder
def identify_quality_column(df):
    for c in ["valid_pixel_fraction","valid_pixel_fraction_any_band","valid_pixel_fraction_all_bands"]:
        if c in df.columns:
            return c
    return None

SOURCE_QUALITY_COLUMN = identify_quality_column(inventory)
print("Source quality column:", SOURCE_QUALITY_COLUMN)

def build_or_reuse(source_files, output_file, band_names):
    output_file = Path(output_file)

    if SKIP_EXISTING_MONTHLY and output_file.exists():
        result = inspect_tiff(output_file, band_names)
        if result.get("reusable", False):
            result.update(
                build_action="skipped_existing",
                composite_created=1,
                file_size_bytes=output_file.stat().st_size
            )
            return result
        try:
            output_file.rename(invalid_backup_path(output_file))
        except Exception:
            output_file.unlink(missing_ok=True)

    try:
        composite, ref = create_monthly_composite(source_files)
        save_composite(composite, ref, output_file)
        result = inspect_tiff(output_file, band_names)
        if not result.get("reusable", False):
            return {"composite_created":0,"build_action":"failed",
                    "valid_pixel_fraction":np.nan,"valid_pixel_percentage":np.nan,
                    "error":result.get("error")}
        result.update(
            composite_created=1,
            build_action="created",
            file_size_bytes=output_file.stat().st_size
        )
        return result
    except Exception as e:
        return {"composite_created":0,"build_action":"failed",
                "valid_pixel_fraction":np.nan,"valid_pixel_percentage":np.nan,
                "error":str(e)}

Source quality column: valid_pixel_fraction


In [10]:
# 10. Fixed site × sensor panel
sensor_master = pd.DataFrame({"sensor":["sentinel1","sentinel2"]})
selected_sample["_k"] = 1
sensor_master["_k"] = 1
site_sensor_master = selected_sample.merge(sensor_master, on="_k").drop(columns="_k")
selected_sample = selected_sample.drop(columns="_k")

if RUN_TEST_ONLY:
    t = selected_sample.loc[selected_sample["group"].eq("treatment"),"site_id"].iloc[0]
    c = selected_sample.loc[selected_sample["group"].eq("counterfactual"),"site_id"].iloc[0]
    site_sensor_to_process = site_sensor_master[
        site_sensor_master["site_id"].isin([str(t),str(c)])
    ].copy()
else:
    site_sensor_to_process = site_sensor_master.copy()

print("Site × sensor combinations:", len(site_sensor_to_process))

Site × sensor combinations: 120


In [11]:
# 11. Generate monthly composites
monthly_records = []

for n, (_, combo) in enumerate(site_sensor_to_process.iterrows(), start=1):
    site_id = str(combo["site_id"])
    group = str(combo["group"])
    sensor = str(combo["sensor"])
    matched = combo["matched_treatment_site_id"]
    rank = combo["control_rank"]

    print(f"\n{n}/{len(site_sensor_to_process)} | {site_id} | {group} | {sensor}")

    site_inv = inventory[
        inventory["site_id"].eq(site_id)
        & inventory["group"].eq(group)
        & inventory["sensor"].eq(sensor)
    ].copy()

    sensor_root = S1_MONTHLY_DIR if sensor == "sentinel1" else S2_MONTHLY_DIR
    bands = SENSOR_BANDS[sensor]

    for _, pr in period_definitions.iterrows():
        phase = pr["period"]
        pid = pr["period_id"]
        month_id = pr["month_id"]
        start = pd.Timestamp(pr["period_start"])
        end = pd.Timestamp(pr["period_end"])

        acquisitions = site_inv[
            site_inv["acquisition_date"].between(start, end)
        ].sort_values("acquisition_date").copy()

        dates = acquisitions["acquisition_date"].dt.strftime("%Y-%m-%d").tolist()
        files = acquisitions["file_path"].astype(str).tolist()
        count = len(files)

        source_mean = np.nan
        source_best = np.nan
        if count and SOURCE_QUALITY_COLUMN:
            q = pd.to_numeric(acquisitions[SOURCE_QUALITY_COLUMN], errors="coerce").dropna()
            if len(q):
                source_mean = float(q.mean())
                source_best = float(q.max())

        record = {
            "site_id":site_id, "group":group,
            "matched_treatment_site_id":matched, "control_rank":rank,
            "sensor":sensor, "period":phase,
            "period_number":int(pr["period_number"]),
            "period_id":pid, "month_id":month_id,
            "period_start":start.strftime("%Y-%m-%d"),
            "period_end":end.strftime("%Y-%m-%d"),
            "calendar_days":int(pr["calendar_days"]),
            "acquisition_count":count,
            "acquisition_dates":json.dumps(dates),
            "source_files":json.dumps(files),
            "has_data":int(count > 0),
            "multiple_acquisitions":int(count > 1),
            "source_mean_valid_pixel_fraction":source_mean,
            "source_best_valid_pixel_fraction":source_best,
            "composite_created":0, "build_action":"missing",
            "output_file":None, "file_size_bytes":np.nan,
            "valid_pixel_fraction":np.nan, "valid_pixel_percentage":np.nan,
            "valid_pixel_fraction_any_band":np.nan,
            "valid_pixel_percentage_any_band":np.nan,
            "valid_pixel_fraction_all_bands":np.nan,
            "valid_pixel_percentage_all_bands":np.nan,
            "quality_label":"missing",
            "improvement_vs_mean_source":np.nan,
            "improvement_vs_best_source":np.nan,
            "error":None,
        }

        print(f"  {pid} {month_id}: {count} acquisitions")

        if count == 0:
            monthly_records.append(record)
            continue

        output_file = (
            sensor_root / group / phase /
            f"{site_id}_{pid}_{month_id}_{start:%Y-%m-%d}_{end:%Y-%m-%d}_monthly_{sensor}.tif"
        )

        result = build_or_reuse(files, output_file, bands)
        record.update(result)

        if result.get("composite_created",0) == 1:
            record["output_file"] = str(output_file)

        if pd.notna(record.get("valid_pixel_fraction")):
            q = float(record["valid_pixel_fraction"])
            record["quality_label"] = quality_label(q)
            if pd.notna(source_mean):
                record["improvement_vs_mean_source"] = q-source_mean
            if pd.notna(source_best):
                record["improvement_vs_best_source"] = q-source_best
            print(f"    valid={q*100:.2f}% | {record['quality_label']} | {record['build_action']}")

        monthly_records.append(record)

monthly_quality = pd.DataFrame(monthly_records)
if monthly_quality.empty:
    raise RuntimeError("No monthly rows generated.")

print("\nCompositing complete.")


1/120 | treatment_0001 | treatment | sentinel1
  M01 2024-05: 2 acquisitions
    valid=100.00% | excellent | created
  M02 2024-06: 2 acquisitions
    valid=100.00% | excellent | created
  M03 2024-07: 3 acquisitions
    valid=100.00% | excellent | created
  M04 2024-08: 2 acquisitions
    valid=100.00% | excellent | created
  M05 2024-09_pre: 2 acquisitions
    valid=100.00% | excellent | created
  M06 2024-09_post: 1 acquisitions
    valid=100.00% | excellent | created
  M07 2024-10: 1 acquisitions
    valid=100.00% | excellent | created
  M08 2024-11: 2 acquisitions
    valid=100.00% | excellent | created
  M09 2024-12: 2 acquisitions
    valid=100.00% | excellent | created
  M10 2025-01: 3 acquisitions
    valid=100.00% | excellent | created
  M11 2025-02: 1 acquisitions
    valid=100.00% | excellent | created

2/120 | treatment_0001 | treatment | sentinel2
  M01 2024-05: 2 acquisitions
    valid=100.00% | excellent | created
  M02 2024-06: 3 acquisitions
    valid=99.55% | excell

In [12]:
# 12. Quality thresholds and detailed CSV
for threshold in [.40,.50,.60,.80,.90]:
    pct = int(threshold*100)
    monthly_quality[f"quality_ge_{pct}pct"] = (
        monthly_quality["valid_pixel_fraction"] >= threshold
    ).astype(int)

monthly_quality["quality_100pct"] = (
    monthly_quality["valid_pixel_fraction"] >= .999999
).astype(int)

monthly_quality.to_csv(QUALITY_CSV_FILE, index=False)
print(QUALITY_CSV_FILE)

/Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/monthly_datasets/monthly_image_quality.csv


In [13]:
# 13. Overall quality summary
monthly_summary = monthly_quality.groupby(
    ["sensor","group","period"], as_index=False
).agg(
    number_of_sites=("site_id","nunique"),
    expected_site_periods=("period_id","count"),
    periods_with_data=("has_data","sum"),
    periods_with_multiple_acquisitions=("multiple_acquisitions","sum"),
    total_acquisitions=("acquisition_count","sum"),
    mean_acquisitions_per_period=("acquisition_count","mean"),
    mean_valid_pixel_fraction=("valid_pixel_fraction","mean"),
    median_valid_pixel_fraction=("valid_pixel_fraction","median"),
    min_valid_pixel_fraction=("valid_pixel_fraction","min"),
    max_valid_pixel_fraction=("valid_pixel_fraction","max"),
    periods_ge_40pct_valid=("quality_ge_40pct","sum"),
    periods_ge_50pct_valid=("quality_ge_50pct","sum"),
    periods_ge_60pct_valid=("quality_ge_60pct","sum"),
    periods_ge_80pct_valid=("quality_ge_80pct","sum"),
    periods_ge_90pct_valid=("quality_ge_90pct","sum"),
    mean_improvement_vs_mean_source=("improvement_vs_mean_source","mean"),
    mean_improvement_vs_best_source=("improvement_vs_best_source","mean"),
)

monthly_summary["periods_without_data"] = (
    monthly_summary["expected_site_periods"]-monthly_summary["periods_with_data"]
)
monthly_summary["percent_periods_with_data"] = (
    monthly_summary["periods_with_data"]/monthly_summary["expected_site_periods"]*100
)

for threshold in [40,50,60,80,90]:
    monthly_summary[f"percent_available_periods_ge_{threshold}pct"] = np.where(
        monthly_summary["periods_with_data"] > 0,
        monthly_summary[f"periods_ge_{threshold}pct_valid"]/
        monthly_summary["periods_with_data"]*100,
        np.nan
    )

for src, dst in [
    ("mean_valid_pixel_fraction","mean_valid_pixel_percentage"),
    ("median_valid_pixel_fraction","median_valid_pixel_percentage"),
    ("min_valid_pixel_fraction","min_valid_pixel_percentage"),
    ("max_valid_pixel_fraction","max_valid_pixel_percentage"),
]:
    monthly_summary[dst] = monthly_summary[src]*100

monthly_summary.to_csv(SUMMARY_CSV_FILE, index=False)
print(monthly_summary.to_string(index=False))

   sensor          group period  number_of_sites  expected_site_periods  periods_with_data  periods_with_multiple_acquisitions  total_acquisitions  mean_acquisitions_per_period  mean_valid_pixel_fraction  median_valid_pixel_fraction  min_valid_pixel_fraction  max_valid_pixel_fraction  periods_ge_40pct_valid  periods_ge_50pct_valid  periods_ge_60pct_valid  periods_ge_80pct_valid  periods_ge_90pct_valid  mean_improvement_vs_mean_source  mean_improvement_vs_best_source  periods_without_data  percent_periods_with_data  percent_available_periods_ge_40pct  percent_available_periods_ge_50pct  percent_available_periods_ge_60pct  percent_available_periods_ge_80pct  percent_available_periods_ge_90pct  mean_valid_pixel_percentage  median_valid_pixel_percentage  min_valid_pixel_percentage  max_valid_pixel_percentage
sentinel1 counterfactual  after               50                    300                297                                 215                 844                      2.813333        

In [14]:
# 14. Period dimension
period_dimension = monthly_quality.groupby(
    ["sensor","group","period","period_number","period_id","month_id","period_start","period_end"],
    as_index=False
).agg(
    number_of_sites=("site_id","nunique"),
    images_with_data=("has_data","sum"),
    mean_valid_pixel_fraction=("valid_pixel_fraction","mean"),
    median_valid_pixel_fraction=("valid_pixel_fraction","median"),
    min_valid_pixel_fraction=("valid_pixel_fraction","min"),
    max_valid_pixel_fraction=("valid_pixel_fraction","max"),
    images_ge_40pct=("quality_ge_40pct","sum"),
    images_ge_50pct=("quality_ge_50pct","sum"),
    images_ge_60pct=("quality_ge_60pct","sum"),
    images_ge_80pct=("quality_ge_80pct","sum"),
    images_ge_90pct=("quality_ge_90pct","sum"),
)

for threshold in [40,50,60,80,90]:
    period_dimension[f"percent_images_ge_{threshold}pct"] = np.where(
        period_dimension["images_with_data"] > 0,
        period_dimension[f"images_ge_{threshold}pct"]/
        period_dimension["images_with_data"]*100,
        np.nan
    )

for src, dst in [
    ("mean_valid_pixel_fraction","mean_valid_pixel_percentage"),
    ("median_valid_pixel_fraction","median_valid_pixel_percentage"),
    ("min_valid_pixel_fraction","min_valid_pixel_percentage"),
    ("max_valid_pixel_fraction","max_valid_pixel_percentage"),
]:
    period_dimension[dst] = period_dimension[src]*100

period_dimension.to_csv(PERIOD_DIMENSION_FILE, index=False)
print("Period dimension saved.")

Period dimension saved.


In [15]:
# 15. Site dimension
site_dimension = monthly_quality.groupby(
    ["sensor","group","site_id","matched_treatment_site_id","control_rank","period"],
    dropna=False, as_index=False
).agg(
    expected_periods=("period_id","count"),
    periods_with_data=("has_data","sum"),
    mean_valid_pixel_fraction=("valid_pixel_fraction","mean"),
    median_valid_pixel_fraction=("valid_pixel_fraction","median"),
    min_valid_pixel_fraction=("valid_pixel_fraction","min"),
    max_valid_pixel_fraction=("valid_pixel_fraction","max"),
    images_ge_40pct=("quality_ge_40pct","sum"),
    images_ge_50pct=("quality_ge_50pct","sum"),
    images_ge_60pct=("quality_ge_60pct","sum"),
    images_ge_80pct=("quality_ge_80pct","sum"),
    images_ge_90pct=("quality_ge_90pct","sum"),
)

site_dimension["periods_without_data"] = site_dimension["expected_periods"]-site_dimension["periods_with_data"]
site_dimension["availability_percentage"] = site_dimension["periods_with_data"]/site_dimension["expected_periods"]*100

for threshold in [40,50,60,80,90]:
    site_dimension[f"percent_images_ge_{threshold}pct"] = np.where(
        site_dimension["periods_with_data"] > 0,
        site_dimension[f"images_ge_{threshold}pct"]/
        site_dimension["periods_with_data"]*100,
        np.nan
    )

for src, dst in [
    ("mean_valid_pixel_fraction","mean_valid_pixel_percentage"),
    ("median_valid_pixel_fraction","median_valid_pixel_percentage"),
    ("min_valid_pixel_fraction","min_valid_pixel_percentage"),
    ("max_valid_pixel_fraction","max_valid_pixel_percentage"),
]:
    site_dimension[dst] = site_dimension[src]*100

site_dimension.to_csv(SITE_DIMENSION_FILE, index=False)
print("Site dimension saved.")

Site dimension saved.


In [16]:
# 16. Build-action and folder summaries
build_action_summary = (
    monthly_quality["build_action"].value_counts(dropna=False)
    .rename_axis("build_action").reset_index(name="image_count")
)
build_action_summary.to_csv(BUILD_ACTION_FILE, index=False)

folder_records = []
for sensor, root in [("sentinel1",S1_MONTHLY_DIR),("sentinel2",S2_MONTHLY_DIR)]:
    for group in ["treatment","counterfactual"]:
        for phase in ["before","after"]:
            folder = root/group/phase
            folder_records.append({
                "sensor":sensor, "group":group, "period":phase,
                "folder":str(folder),
                "monthly_tiff_count":len(list(folder.glob("*.tif")))
            })

folder_summary = pd.DataFrame(folder_records)
folder_summary.to_csv(FOLDER_SUMMARY_FILE, index=False)

print(build_action_summary.to_string(index=False))

build_action  image_count
     created         1254
     missing           66


In [17]:
# 17. Excel workbook
quality_definitions = pd.DataFrame({
    "variable":[
        "period_id","month_id","acquisition_count","has_data",
        "valid_pixel_fraction","valid_pixel_fraction_any_band",
        "valid_pixel_fraction_all_bands","build_action"
    ],
    "meaning":[
        "Sequential monthly analysis period.",
        "Calendar month; September is split pre/post Helene.",
        "Number of daily/acquisition TIFFs contributing to composite.",
        "1 if at least one source acquisition exists.",
        "Primary: fraction of pixels with at least one finite output band.",
        "Explicit any-band version of primary metric.",
        "Strict diagnostic requiring all output bands finite.",
        "created, skipped_existing, missing, or failed."
    ]
})

try:
    with pd.ExcelWriter(QUALITY_EXCEL_FILE, engine="openpyxl") as writer:
        monthly_quality.to_excel(writer, sheet_name="image_quality", index=False)
        monthly_summary.to_excel(writer, sheet_name="summary", index=False)
        period_dimension.to_excel(writer, sheet_name="period_dimension", index=False)
        site_dimension.to_excel(writer, sheet_name="site_dimension", index=False)
        period_definitions.to_excel(writer, sheet_name="period_definitions", index=False)
        selected_sample.to_excel(writer, sheet_name="selected_sample", index=False)
        sample_validation.to_excel(writer, sheet_name="sample_validation", index=False)
        quality_definitions.to_excel(writer, sheet_name="quality_definitions", index=False)
        build_action_summary.to_excel(writer, sheet_name="build_actions", index=False)
    print("Excel saved:", QUALITY_EXCEL_FILE)
except ModuleNotFoundError:
    print("openpyxl not installed; CSV outputs were still generated.")

Excel saved: /Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/monthly_datasets/monthly_image_quality.xlsx


In [18]:
# 18. Final validation and Sentinel-2 quality display
total_sites = selected_sample["site_id"].nunique()
period_count = len(period_definitions)
expected_per_sensor = total_sites * period_count
expected_both = expected_per_sensor * 2
actual = len(monthly_quality)

coverage = monthly_quality.groupby(["sensor","group"], as_index=False).agg(
    unique_sites=("site_id","nunique"),
    total_site_period_rows=("period_id","count"),
    periods_with_data=("has_data","sum"),
    composites_available=("composite_created","sum")
)

print("="*100)
print("MONTHLY PANEL VALIDATION")
print("="*100)
print(coverage.to_string(index=False))
print("\nPeriods:", period_count)
print("Expected rows per sensor:", expected_per_sensor)
print("Expected rows both sensors:", expected_both)
print("Actual rows:", actual)

s2 = period_dimension[period_dimension["sensor"].eq("sentinel2")].copy()
print("\n" + "="*100)
print("SENTINEL-2 MONTHLY QUALITY")
print("="*100)
print(s2[[
    "group","period","period_id","month_id","period_start","period_end",
    "images_with_data","mean_valid_pixel_percentage",
    "median_valid_pixel_percentage","images_ge_80pct",
    "percent_images_ge_80pct"
]].to_string(index=False))

MONTHLY PANEL VALIDATION
   sensor          group  unique_sites  total_site_period_rows  periods_with_data  composites_available
sentinel1 counterfactual            50                     550                544                   544
sentinel1      treatment            10                     110                110                   110
sentinel2 counterfactual            50                     550                500                   500
sentinel2      treatment            10                     110                100                   100

Periods: 11
Expected rows per sensor: 660
Expected rows both sensors: 1320
Actual rows: 1320

SENTINEL-2 MONTHLY QUALITY
         group period period_id     month_id period_start period_end  images_with_data  mean_valid_pixel_percentage  median_valid_pixel_percentage  images_ge_80pct  percent_images_ge_80pct
counterfactual  after       M06 2024-09_post   2024-09-27 2024-09-30                 0                          NaN                            N

In [19]:
# 19. Final summary
print("\n" + "="*100)
print("NOTEBOOK 13 COMPLETE")
print("="*100)
print("Source:", DAILY_DIR)
print("Output:", MONTHLY_DIR)
print("\nTemporal periods:")
print(period_definitions[[
    "period_id","month_id","period","period_start","period_end","calendar_days"
]].to_string(index=False))
print("\nSeptember is split at 2024-09-27; no composite crosses the Helene cutoff.")
print("\nNo Earth Engine request or new download occurs.")
print("\nQuality CSV:", QUALITY_CSV_FILE)
print("Quality Excel:", QUALITY_EXCEL_FILE)
print("Period definitions:", PERIOD_DEFINITION_FILE)
print("\nNotebook completed successfully.")


NOTEBOOK 13 COMPLETE
Source: /Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/daily_datasets
Output: /Users/gaoyujuan/REAP Dropbox/Gao yujuan/Virginia Tech/CALS/datasets/finals/monthly_datasets

Temporal periods:
period_id     month_id period period_start period_end  calendar_days
      M01      2024-05 before   2024-05-10 2024-05-31             22
      M02      2024-06 before   2024-06-01 2024-06-30             30
      M03      2024-07 before   2024-07-01 2024-07-31             31
      M04      2024-08 before   2024-08-01 2024-08-31             31
      M05  2024-09_pre before   2024-09-01 2024-09-26             26
      M06 2024-09_post  after   2024-09-27 2024-09-30              4
      M07      2024-10  after   2024-10-01 2024-10-31             31
      M08      2024-11  after   2024-11-01 2024-11-30             30
      M09      2024-12  after   2024-12-01 2024-12-31             31
      M10      2025-01  after   2025-01-01 2025-01-31             31
